# Lab 01: FashionMNIST Classification với PyTorch
> **Môn học:** Deep Learning Labs  
> **Kiến trúc:** Clean Architecture & Modular `src/` (Chạy được cả Local và Kaggle GPU)  
> **Tài liệu tham chiếu:** `dl_workflows/DL_flow.md` | `dl_workflows/dl_training_and_optimization_guide.md`

---

## Mục Tiêu Bài Thực Hành
1. **Learn (Nghiên cứu lý thuyết):**
   - PyTorch Tensors, Autograd & Đồ thị tính toán.
   - Quản lý dữ liệu với `Dataset` và `DataLoader` (Lazy loading, Batching, Pin memory).
   - Torchvision `transforms` & Chiến lược Data Augmentation.
   - Xây dựng mô hình với `nn.Module` (Linear, Conv2d, BatchNorm, Dropout).
   - Vòng lặp huấn luyện chuẩn: Forward pass $\to$ Loss calculation $\to$ Backward pass $\to$ Optimizer step.
   - Lưu trữ và phục hồi trạng thái mô hình (`torch.save` & `torch.load` với `state_dict`).
2. **Implement (Hiện thực hóa Pipeline từ `src/`):**
   - Nạp tập dữ liệu FashionMNIST (60,000 ảnh train, 10,000 ảnh test, 10 lớp phân loại).
   - Chuẩn hóa theo phân phối thực tế ($\mu = 0.2860, \sigma = 0.3205$).
   - Kế thừa và sử dụng các module chuẩn từ thư mục `src/`:
     - `src.config`: Cấu hình siêu tham số, thiết bị tính toán, seed.
     - `src.dataset`: Xây dựng Custom Dataset, Transforms, DataLoaders.
     - `src.model`: Kiến trúc BaselineMLP, TunedMLP, FashionCNN.
     - `src.engine`: Vòng lặp train, eval, sanity overfit.
     - `src.utils`: Seed, lưu/nạp checkpoint, trực quan hóa biểu đồ.
   - Thực hiện kiểm thử TDD Sanity Overfit trên 1 batch nhỏ trước khi huấn luyện diện rộng.
3. **Experiment & Analyze (Thực nghiệm & Phân tích):**
   - So sánh 3 phiên bản: **Run #1 (Baseline MLP)** vs **Run #2 (Tuned MLP)** vs **Run #3 (FashionCNN)**.
   - Trực quan hóa đường cong Loss & Accuracy theo epoch.
   - Hiển thị dự đoán Predicted vs Actual với mã màu trực quan (Xanh: Đúng, Đỏ: Sai).
   - Khảo sát Ma trận nhầm lẫn (Confusion Matrix) và phân tích các trường hợp khó phân loại.

## PHẦN 1: TỔNG QUAN LÝ THUYẾT PYTORCH CỐT LÕI

Dưới đây là tóm tắt cô đọng 7 trụ cột công nghệ trong PyTorch được áp dụng trong bài Lab:
1. **PyTorch Tensor:** Cấu trúc mảng đa chiều hỗ trợ tính toán song song trên GPU và tự động theo dõi đạo hàm (`requires_grad=True`).
2. **Autograd:** Động cơ vi phân tự động (Automatic Differentiation) xây dựng đồ thị tính toán động qua mỗi forward pass và lan truyền ngược gradient khi gọi `.backward()`.
3. **Dataset & DataLoader:**
   - `Dataset`: Định nghĩa cách truy xuất một mẫu dữ liệu qua `__len__()` và `__getitem__()`.
   - `DataLoader`: Đóng gói việc gom nhóm mini-batch, xáo trộn dữ liệu (`shuffle=True` trên tập train), nạp đa tiến trình, và khóa trang bộ nhớ (`pin_memory=True`).
4. **Transforms:** Tiền xử lý tensor và tăng cường dữ liệu ngẫu nhiên nhằm nâng cao tính bất biến (invariance) của mô hình.
5. **Model Building (`nn.Module`):** Mọi kiến trúc mạng kế thừa `nn.Module`, định nghĩa các tầng trong `__init__()` và luồng tính toán trong `forward()`.
6. **Loss & Optimizer:** `nn.CrossEntropyLoss` kết hợp LogSoftmax và NLLLoss; `AdamW` phân tách trọng số suy giảm (weight decay) khỏi moment động lượng.
7. **Model Checkpointing:** Lưu trữ `model.state_dict()` để đảm bảo tính di động và an toàn mã nguồn.

In [ ]:
# Minh họa nhanh 7 khái niệm cốt lõi trong PyTorch
import torch
import torch.nn as nn

# 1. Tensor & Autograd
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()
y.backward()
print("Tensor x:", x)
print("Đạo hàm dy/dx (2 * x):", x.grad)

# 2. Xây dựng Module tối thiểu
class ToyLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)
    def forward(self, t):
        return self.fc(t)

toy = ToyLinear()
print("Toy model parameters:", list(toy.parameters()))

## PHẦN 2: THIẾT LẬP MÔI TRƯỜNG & IMPORT TỪ THƯ MỤC `src/`

Đoạn code dưới đây tự động nhận diện môi trường thực thi:
- Nếu đang chạy tại **Local**: Nhận diện thư mục `../src` ngay trên máy bạn.
- Nếu đang chạy trên **Kaggle GPU**: Tự động tải repository `Lab-DL` từ GitHub về `/kaggle/working/Lab-DL` và import trực tiếp `src/`!

In [ ]:
import os
import sys
import shutil
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np

# 1. Don dep triet de cache va duong dan cu trong bo nho Kernel
sys.path = [p for p in sys.path if not p.startswith("/kaggle/input")]
for m in list(sys.modules.keys()):
    if m == "src" or m.startswith("src."):
        del sys.modules[m]

# 2. Nhan dien moi truong Kaggle vs Local
if os.path.exists("/kaggle/working"):
    print("[INFO] Phat hien dang chay tren moi truong Kaggle!")
    
    # Tim thu muc src trong Kaggle Input (Dataset)
    input_src_list = list(Path("/kaggle/input").glob("**/src"))
    if input_src_list:
        print(f"[INFO] Tim thay src tu Kaggle Input: {input_src_list[0]}")
        # Sao chep sang /kaggle/working/src de co toan quyen Ghi/Luu model
        target_src = Path("/kaggle/working/src")
        if target_src.exists():
            shutil.rmtree(target_src)
        shutil.copytree(input_src_list[0], target_src)
        
        # Dam bao config.py luon dung /kaggle/working lam thu muc goc
        cfg_file = target_src / "config.py"
        if cfg_file.exists():
            content = cfg_file.read_text(encoding="utf-8")
            content = content.replace("BASE_DIR = Path(__file__).resolve().parent.parent", "BASE_DIR = Path('/kaggle/working')")
            cfg_file.write_text(content, encoding="utf-8")
    else:
        # Neu khong co trong Input, clone tu GitHub
        repo_dir = Path("/kaggle/working/Lab-DL")
        if not (repo_dir / "labs" / "lab01" / "src").exists():
            if repo_dir.exists():
                shutil.rmtree(repo_dir, ignore_errors=True)
            print("[INFO] Dang clone repository Lab-DL tu GitHub...")
            !git clone https://github.com/lacthui06/Lab-DL.git /kaggle/working/Lab-DL
        
        if (repo_dir / "labs" / "lab01" / "src").exists():
            target_src = Path("/kaggle/working/src")
            if target_src.exists():
                shutil.rmtree(target_src)
            shutil.copytree(repo_dir / "labs" / "lab01" / "src", target_src)

    # Dat /kaggle/working len vi tri uu tien so 1
    if "/kaggle/working" in sys.path:
        sys.path.remove("/kaggle/working")
    sys.path.insert(0, "/kaggle/working")
    PROJECT_ROOT = Path("/kaggle/working")
else:
    print("[INFO] Phat hien dang chay tren moi truong Local!")
    PROJECT_ROOT = Path("..").resolve()
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

# 3. Import truc tiep tu cac module trong src/
from src.config import (
    DATA_DIR,
    OUTPUTS_DIR,
    CHECKPOINTS_DIR,
    DEVICE,
    RANDOM_SEED,
    CLASS_NAMES,
    FASHION_MNIST_MEAN,
    FASHION_MNIST_STD,
    RUN1_CONFIG,
    RUN2_CONFIG,
    RUN3_CONFIG,
)
from src.utils import (
    set_seed,
    plot_loss_and_acc_curves,
    plot_prediction_grid,
    plot_confusion_matrix,
    plot_comparison_curves,
)
from src.dataset import get_dataloaders
from src.model import build_model, count_parameters
from src.engine import train_model, evaluate, sanity_overfit_check

# 4. Co dinh ngau nhien (Reproducibility)
set_seed(RANDOM_SEED)

print(f"\n[INFO] Thiet bi tinh toan: {DEVICE}")
if torch.cuda.is_available():
    print(f"[INFO] Tang toc GPU: {torch.cuda.get_device_name(0)}")
print(f"[INFO] Thu muc Project Root: {PROJECT_ROOT}")
print(f"[INFO] Thu muc Outputs: {OUTPUTS_DIR}")
print(f"[INFO] Danh sach 10 lop: {CLASS_NAMES}")


## PHẦN 3: NẠP DỮ LIỆU & KHÁM PHÁ (DL EDA)

FashionMNIST gồm các ảnh đen trắng $28 \times 28$ thuộc 10 danh mục trang phục.  
Dữ liệu được nạp qua hàm `get_dataloaders()` từ module `src.dataset`, chia thành 3 phần độc lập:
- **Train Set:** 54,000 ảnh (dùng để cập nhật gradient)
- **Validation Set:** 6,000 ảnh (dùng để chọn hyperparameter và lưu checkpoint)
- **Test Set:** 10,000 ảnh (độc lập hoàn toàn, dùng để nghiệm thu cuối cùng)

In [ ]:
# Khởi tạo DataLoaders không có augmentation cho khâu khảo sát EDA
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=64,
    val_split=0.1,
    augment=False,
    seed=RANDOM_SEED,
)

print(f"Số batch Train: {len(train_loader)} (tổng ~{len(train_loader) * 64} mẫu)")
print(f"Số batch Validation: {len(val_loader)} (tổng ~{len(val_loader) * 64} mẫu)")
print(f"Số batch Test: {len(test_loader)} (tổng ~{len(test_loader) * 64} mẫu)")

# Trực quan hóa một batch mẫu đầu tiên từ tập Train
sample_images, sample_targets = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
axes = axes.flatten()
for i in range(16):
    img = sample_images[i].squeeze().numpy() * FASHION_MNIST_STD[0] + FASHION_MNIST_MEAN[0]
    img = np.clip(img, 0.0, 1.0)
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(class_names[sample_targets[i].item()], fontsize=9, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("Các mẫu ảnh minh họa trong FashionMNIST Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## PHẦN 4: TEST-DRIVEN DEVELOPMENT — KIỂM THỬ SANITY OVERFIT 1 BATCH

Tuân thủ **Nguyên tắc số 4 trong `AGENTS.md`**: Trước khi huấn luyện toàn bộ tập dữ liệu, bắt buộc phải kiểm tra xem mô hình có thể ép Loss về gần 0 và đạt Accuracy 100% trên một batch nhỏ (8 mẫu) hay không (gọi hàm `sanity_overfit_check` từ `src.engine`).  
Nếu không thể overfit 8 mẫu, kiến trúc hoặc hàm loss chắc chắn có lỗi!

In [ ]:
# Lấy 1 mini-batch nhỏ gồm 8 mẫu
test_images_small = sample_images[:8]
test_targets_small = sample_targets[:8]

toy_model = build_model("baseline_mlp")
passed, final_loss, final_acc = sanity_overfit_check(
    model=toy_model,
    sample_images=test_images_small,
    sample_targets=test_targets_small,
    device=DEVICE,
    num_epochs=40,
    lr=1e-2,
)

print(f"Sanity Overfit Test: {'[PASSED] ĐẠT YÊU CẦU' if passed else '[FAILED] KHÔNG ĐẠT'}")
print(f"Loss sau 40 epoch trên 8 mẫu: {final_loss:.4f} | Accuracy: {final_acc:.2f}%")
assert passed, "Mô hình không thể học trên 8 mẫu!" 

## PHẦN 5: KHỞI TẠO CÁC MÔ HÌNH TỪ `src.model`

Sử dụng hàm nhà máy `build_model` từ `src.model` để khởi tạo 3 kiến trúc:
1. **Run #1 — BaselineMLP:** Mạng nơ-ron truyền thẳng 2 tầng (101,770 tham số).
2. **Run #2 — TunedMLP:** Mạng nơ-ron sâu 3 tầng bổ sung Batch Normalization và Dropout (235,914 tham số).
3. **Run #3 — FashionCNN:** Mạng nơ-ron tích chập 2 khối Conv (32 và 64 filter) có khả năng bảo toàn không gian 2D (422,090 tham số).

In [ ]:
models = {
    "Baseline MLP": build_model("baseline_mlp"),
    "Tuned MLP (BN + Dropout)": build_model("tuned_mlp"),
    "FashionCNN (2 Conv Blocks)": build_model("cnn"),
}

print("=" * 65)
print(f"{'Tên Mô Hình':<30} | {'Tổng Tham Số':<15} | {'Dung lượng (MB)':<12}")
print("=" * 65)
for name, m in models.items():
    param_info = count_parameters(m)
    param_mb = (param_info['total'] * 4) / (1024 * 1024)
    print(f"{name:<30} | {param_info['total']:<15,d} | {param_mb:.2f} MB")
print("=" * 65)

## PHẦN 6: THỰC THI 3 PHIÊN THỬ NGHIỆM QUA `src.engine`

- **Run #1 (Baseline):** 5 epochs, AdamW (lr=1e-3), không scheduler, không augmentation.
- **Run #2 (Tuned MLP):** 10 epochs, AdamW (lr=1e-3, weight_decay=1e-4), CosineAnnealingLR, RandomHorizontalFlip.
- **Run #3 (FashionCNN):** 10 epochs, AdamW, CosineAnnealingLR, RandomHorizontalFlip.

In [ ]:
# Nạp DataLoaders có Data Augmentation cho Run #2 và Run #3
train_loader_aug, _, _, _ = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=64,
    val_split=0.1,
    augment=True,
    seed=RANDOM_SEED,
)

criterion = nn.CrossEntropyLoss()
histories = {}
test_results = {}

experiments = [
    (RUN1_CONFIG, train_loader, "baseline_mlp"),
    (RUN2_CONFIG, train_loader_aug, "tuned_mlp"),
    (RUN3_CONFIG, train_loader_aug, "cnn"),
]

for cfg, loader, m_type in experiments:
    print(f"\n>>> Đang chạy thử nghiệm: {cfg.name}...")
    model = build_model(m_type)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.num_epochs, eta_min=1e-5) if cfg.use_scheduler else None
    chk_path = CHECKPOINTS_DIR / cfg.checkpoint_name
    
    hist, best_val = train_model(
        model=model,
        train_loader=loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=cfg.num_epochs,
        device=DEVICE,
        save_path=chk_path,
        verbose=True,
    )
    histories[cfg.name] = hist
    
    # Nạp checkpoint tốt nhất và kiểm thử trên tập Test
    chk = torch.load(str(chk_path), map_location=DEVICE)
    model.load_state_dict(chk["model_state_dict"])
    t_loss, t_acc, t_preds, t_targets = evaluate(model, test_loader, criterion, DEVICE)
    test_results[cfg.name] = {"test_acc": t_acc, "test_loss": t_loss, "preds": t_preds, "targets": t_targets, "model": model}
    print(f"[RESULT] Kết quả Test của {cfg.name}: Loss = {t_loss:.4f} | Accuracy = {t_acc:.2f}%")

## PHẦN 7: TRỰC QUAN HÓA & SO SÁNH ĐƯỜNG CONG HỌC TẬP

Sử dụng `plot_comparison_curves` và `plot_loss_and_acc_curves` từ `src.utils` để so sánh và xuất biểu đồ ra thư mục `outputs/`.

In [ ]:
# 1. So sánh đối đầu giữa cả 3 Runs
plot_comparison_curves(histories, save_path=OUTPUTS_DIR / "all_models_comparison_curves.png")

# 2. Chi tiết từng Run
plot_loss_and_acc_curves(histories[RUN1_CONFIG.name], title="Run 1: Baseline MLP", save_path=OUTPUTS_DIR / "run1_baseline_curves.png")
plot_loss_and_acc_curves(histories[RUN2_CONFIG.name], title="Run 2: Tuned MLP", save_path=OUTPUTS_DIR / "run2_tuned_mlp_curves.png")
plot_loss_and_acc_curves(histories[RUN3_CONFIG.name], title="Run 3: FashionCNN", save_path=OUTPUTS_DIR / "run3_cnn_curves.png")

## PHẦN 8: CHẨN ĐOÁN LỖI & HIỂN THỊ DỰ ĐOÁN (PREDICTED VS. ACTUAL)

Trực quan hóa ma trận nhầm lẫn (`plot_confusion_matrix`) và hiển thị lưới 16 ảnh kiểm thử ngẫu nhiên (`plot_prediction_grid`).  
Mã màu trực quan:  
- **Xanh lá (Green):** Dự đoán chính xác (Correct).  
- **Đỏ (Red):** Dự đoán sai (Misclassified) kèm xác suất tự tin của mô hình.

In [ ]:
best_cnn_res = test_results["Run3_FashionCNN"]

# 1. Ma trận nhầm lẫn chuẩn hóa
plot_confusion_matrix(
    y_true=best_cnn_res["targets"],
    y_pred=best_cnn_res["preds"],
    class_names=class_names,
    title="FashionCNN: Normalized Test Confusion Matrix",
    save_path=OUTPUTS_DIR / "confusion_matrix_cnn.png",
)

# 2. Lưới ảnh dự đoán kèm nhãn thực tế
plot_prediction_grid(
    model=best_cnn_res["model"],
    dataloader=test_loader,
    class_names=class_names,
    device=DEVICE,
    num_images=16,
    save_path=OUTPUTS_DIR / "predictions_cnn.png",
)

## PHẦN 9: XÁC THỰC LƯU VÀ NẠP MÔ HÌNH (SAVE & LOAD VERIFICATION)

Trình bày quy trình nạp lại mô hình từ file checkpoint và kiểm tra tính nhất quán 100% của đầu ra suy luận.

In [ ]:
from src.utils import save_checkpoint, load_checkpoint

# 1. Nạp lại mô hình từ file
loaded_model = build_model("cnn")
load_checkpoint(loaded_model, CHECKPOINTS_DIR / "fashion_cnn_best.pt", device=DEVICE)
loaded_model.eval()

# 2. Kiểm tra tính tương đương của kết quả suy luận
sample_img = test_images_small[0:1]
with torch.no_grad():
    orig_output = best_cnn_res["model"](sample_img.to(DEVICE))
    loaded_output = loaded_model(sample_img.to(DEVICE))

assert torch.allclose(orig_output, loaded_output, atol=1e-6), "Mô hình nạp lại cho kết quả sai lệch so với mô hình gốc!"
print("[PASSED] Xác thực thành công: Kết quả suy luận của mô hình trước và sau khi lưu là hoàn toàn trùng khớp 100%!")

## PHẦN 10: TỰ ĐỘNG ĐÓNG GÓI ARTIFACTS ĐỂ TẢI VỀ MÁY LOCAL

Cell dưới đây tự động nén toàn bộ kết quả trong thư mục `outputs/` thành file **`outputs.zip`**.  
Khi chạy trên Kaggle, bạn chỉ cần bấm vào tab **Output** bên phải màn hình để tải file `outputs.zip` về máy!

In [ ]:
import json
import shutil
from sklearn.metrics import classification_report

# Luu file ket qua tong hop JSON
summary_data = {
    cfg.name: {
        "test_loss": round(test_results[cfg.name]["test_loss"], 4),
        "test_acc": round(test_results[cfg.name]["test_acc"], 2),
    } for cfg, _, _ in experiments
}
with open(OUTPUTS_DIR / "experiment_results.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

with open(OUTPUTS_DIR / "classification_report_cnn.txt", "w", encoding="utf-8") as f:
    f.write(classification_report(best_cnn_res["targets"], best_cnn_res["preds"], target_names=class_names, digits=4))

print("[INFO] Du lieu tong hop:")
print(json.dumps(summary_data, indent=2))

# Neu chay tren Kaggle: Nen toan bo thu muc outputs thanh file zip de download
if os.path.exists("/kaggle/working"):
    zip_path = shutil.make_archive("/kaggle/working/outputs", "zip", OUTPUTS_DIR)
    print(f"\n[SUCCESS] File {zip_path} da duoc tao thanh cong!")
    print("[INSTRUCTION] Hay nhin sang cot ben phai tai muc 'Output', bam vao dau 3 cham canh outputs.zip -> chon 'Download' ve may!")
else:
    print("\n[SUCCESS] Chay tren Local hoan tat! Toan bo file da duoc luu trong thu muc outputs/.")
